In [0]:
# ================================================================
# NOTEBOOK: nb_silver_returns_cdc_merge
# PURPOSE:  ADF CDC Parquet → Silver Merge for Returns
#
# SOURCE:   bronze/cdc/returns/
# TARGET:   silver/returns/
#
# HANDLES:
#   INSERT → Insert into Silver
#   UPDATE → Update existing Silver row
#
# LIMITATION:
#   Physical DELETE is not handled because Bronze Parquet does not
#   contain CDC operation metadata such as __$operation.
# ================================================================

from pyspark.sql import functions as F
from pyspark.sql.functions import col, to_timestamp, upper, trim
from pyspark.sql.window import Window
from delta.tables import DeltaTable


CDC_PATH = ("abfss://source@stshopsensedevhj.dfs.core.windows.net/bronze/cdc/returns/")
SILVER_PATH = ("abfss://source@stshopsensedevhj.dfs.core.windows.net/silver/returns/")


# ================================================================
# STEP 1: READ BRONZE
# ================================================================

cdc_raw = spark.read.parquet(CDC_PATH)

total_rows = cdc_raw.count()

if total_rows == 0:
    print("[INFO] No Returns data found. Exiting.")
    dbutils.notebook.exit("NO_CHANGES")

print(f"[BRONZE] Rows read: {total_rows}")
print("[BRONZE COLUMNS]")
print(cdc_raw.columns)


# ================================================================
# STEP 2: KEEP LATEST ROW PER ReturnID
# ================================================================

latest_window = (
    Window
    .partitionBy("ReturnID")
    .orderBy(F.desc("LastModifiedDate"))
)

cdc_latest = (
    cdc_raw
    .withColumn("_rn", F.row_number().over(latest_window))
    .filter(col("_rn") == 1)
    .drop("_rn")
)

latest_count = cdc_latest.count()

print(f"[LATEST] Latest ReturnID rows: {latest_count}")


# ================================================================
# STEP 3: TRANSFORM RETURNS
# ================================================================

def transform_returns(df):
    return (
        df

        .withColumn(
            "ReturnDate",
            to_timestamp(col("ReturnDate"))
        )
        .withColumn(
            "RefundDate",
            to_timestamp(col("RefundDate"))
        )
        .withColumn(
            "LastModifiedDate",
            to_timestamp(col("LastModifiedDate"))
        )

        .withColumn(
            "RefundAmount",
            col("RefundAmount").cast("decimal(10,2)")
        )

        .withColumn(
            "ReturnStatus",
            upper(trim(col("ReturnStatus")))
        )
        .withColumn(
            "ReturnReason",
            upper(trim(col("ReturnReason")))
        )
        .withColumn(
            "RefundMethod",
            upper(trim(col("RefundMethod")))
        )
        .withColumn(
            "ConditionOnReturn",
            upper(trim(col("ConditionOnReturn")))
        )

        .withColumn(
            "IsApproved",
            col("ReturnStatus") == "REFUNDED"
        )
        .withColumn(
            "IsRejected",
            col("ReturnStatus") == "REJECTED"
        )
        .withColumn(
            "IsPending",
            col("ReturnStatus") == "PROCESSING"
        )
        .withColumn(
            "ReturnYear",
            F.year(col("ReturnDate"))
            )
        
        .withColumn(
            "ReturnMonth",
            F.month(col("ReturnDate"))
            )
        
        .withColumn(
            "IsDamaged",
            col("ConditionOnReturn") == "DAMAGED"
            )
        
        
        .withColumn(
            "ReasonCategory",
            F.when(
                col("ReturnReason").isin(
                    "DEFECTIVE_PRODUCT",
                    "QUALITY_ISSUE",
                    "NOT_AS_DESCRIBED"
                    ),
                "PRODUCT_ISSUE"
                )
            .when(
                col("ReturnReason") == "WRONG_ITEM",
                "FULFILLMENT_ISSUE"
                )
            .when(
                col("ReturnReason") == "CHANGED_MIND",
                "CUSTOMER_PREFERENCE"
                )
            .otherwise("OTHER")
            )
        .withColumn(
            "IsQuickReturn",
            F.datediff(
                col("ReturnDate"),
                col("ReturnDate")
                ) <= 7
            )
        .withColumn(
            "DaysToProcess",
            F.when(
                col("RefundDate").isNotNull(),
                F.datediff(
                    col("RefundDate"),
                    col("ReturnDate")
                )
            ).otherwise(F.lit(None))
        )

        .withColumn(
            "_silver_load_ts",
            F.current_timestamp()
        )
        .withColumn(
            "source",
            F.lit("adf_cdc_parquet")
        )
        .withColumn(
            "_is_deleted",
            F.lit(False)
        )
    )


transformed_returns = transform_returns(cdc_latest)


# ================================================================
# STEP 4: CHECK SILVER TABLE
# ================================================================

if not DeltaTable.isDeltaTable(spark, SILVER_PATH):
    raise Exception(
        "Silver Returns Delta table does not exist. "
        "Run the initial Silver Returns load first."
    )

silver_df = (
    spark.read
    .format("delta")
    .load(SILVER_PATH)
)

silver = DeltaTable.forPath(
    spark,
    SILVER_PATH
)


# ================================================================
# STEP 5: SCHEMA CHECK
# ================================================================

source_columns = set(transformed_returns.columns)
target_columns = set(silver_df.columns)

missing_in_source = sorted(
    target_columns - source_columns
)

extra_in_source = sorted(
    source_columns - target_columns
)

print("[SCHEMA CHECK] Missing in transformed source:")
print(missing_in_source)

print("[SCHEMA CHECK] Extra in transformed source:")
print(extra_in_source)

if missing_in_source:
    raise Exception(
        "Merge stopped because these Silver columns are missing "
        f"from transformed Returns: {missing_in_source}"
    )


transformed_returns = transformed_returns.select(
    *silver_df.columns
)


# ================================================================
# STEP 6: MERGE INSERTS + UPDATES
# ================================================================

(
    silver.alias("s")
    .merge(
        transformed_returns.alias("c"),
        "s.ReturnID = c.ReturnID"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print(
    f"[MERGE] {latest_count} Returns merged into Silver."
)


# ================================================================
# STEP 7: VERIFY
# ================================================================

final_df = (
    spark.read
    .format("delta")
    .load(SILVER_PATH)
)

total_count = final_df.count()

active_count = (
    final_df
    .filter(col("_is_deleted") == False)
    .count()
)

deleted_count = (
    final_df
    .filter(col("_is_deleted") == True)
    .count()
)

print(
    "[DONE] Silver Returns:"
    f"\nTotal        : {total_count}"
    f"\nActive       : {active_count}"
    f"\nSoft-deleted : {deleted_count}"
)

[BRONZE] Rows read: 263
[BRONZE COLUMNS]
['ReturnID', 'OrderID', 'CustomerID', 'ReturnDate', 'ReturnReason', 'ReturnStatus', 'RefundAmount', 'RefundDate', 'RefundMethod', 'ConditionOnReturn', 'LastModifiedDate']
[LATEST] Latest ReturnID rows: 257
[SCHEMA CHECK] Missing in transformed source:
[]
[SCHEMA CHECK] Extra in transformed source:
[]
[MERGE] 257 Returns merged into Silver.
[DONE] Silver Returns:
Total        : 409
Active       : 409
Soft-deleted : 0


In [0]:
display(
    final_df
    .filter(
        col("ReturnID").isin(
            "RET_CDC_101",
            "RET_CDC_102",
            "RET_CDC_103",
            "RET_CDC_104",
            "RET_CDC_105"
        )
    )
    .select(
        "ReturnID",
        "ReturnStatus",
        "RefundAmount",
        "IsApproved",
        "IsRejected",
        "IsPending",
        "_is_deleted",
        "LastModifiedDate"
    )
    .orderBy("ReturnID")
)

ReturnID,ReturnStatus,RefundAmount,IsApproved,IsRejected,IsPending,_is_deleted,LastModifiedDate
RET_CDC_101,REFUNDED,1200.00,true,false,false,false,2026-07-15T17:36:10.280Z
RET_CDC_102,REFUNDED,1500.00,true,false,false,false,2026-07-15T17:36:10.280Z
RET_CDC_103,REJECTED,800.00,false,true,false,false,2026-07-15T17:36:31.370Z
RET_CDC_104,REJECTED,2000.00,false,true,false,false,2026-07-15T17:36:31.370Z
RET_CDC_105,PROCESSING,2550.00,false,false,true,false,2026-07-15T17:36:57.690Z
